# TTL Scheduler GUI

Run the next cell to start the PySide6/PyQt6 GUI.

In [ ]:
import csv
import json
import math
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Optional

import serial
from serial import SerialException

try:
    from PySide6.QtCore import QTime, QTimer, Qt
    from PySide6.QtGui import QColor, QPainter, QPalette, QPen
    from PySide6.QtWidgets import (
        QApplication,
        QCheckBox,
        QComboBox,
        QDoubleSpinBox,
        QFileDialog,
        QFormLayout,
        QGridLayout,
        QGroupBox,
        QHBoxLayout,
        QLabel,
        QLineEdit,
        QMainWindow,
        QMessageBox,
        QPushButton,
        QSpinBox,
        QTextEdit,
        QTimeEdit,
        QVBoxLayout,
        QWidget,
    )
    QT_API = "PySide6"
except ImportError:
    from PyQt6.QtCore import QTime, QTimer, Qt
    from PyQt6.QtGui import QColor, QPainter, QPalette, QPen
    from PyQt6.QtWidgets import (
        QApplication,
        QCheckBox,
        QComboBox,
        QDoubleSpinBox,
        QFileDialog,
        QFormLayout,
        QGridLayout,
        QGroupBox,
        QHBoxLayout,
        QLabel,
        QLineEdit,
        QMainWindow,
        QMessageBox,
        QPushButton,
        QSpinBox,
        QTextEdit,
        QTimeEdit,
        QVBoxLayout,
        QWidget,
    )
    QT_API = "PyQt6"


MIN_FREQ_HZ = 1.0 / 300.0
MAX_INTERVAL_MS = 300_000


@dataclass
class TimingConfig:
    mode_name: str
    period_us: int
    pulse_us: int
    count: int
    freq_hz: float
    interval_ms: float
    total_duration_s: float
    low_phase_ms: float
    now_str: str
    start_str: str
    classic_arm_compatible: bool


def seconds_to_hms(total_seconds: float) -> str:
    total_seconds = max(0, int(round(total_seconds)))
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def format_interval_ms(total_ms: int) -> str:
    total_ms = max(0, int(total_ms))
    hours = total_ms // 3_600_000
    minutes = (total_ms % 3_600_000) // 60_000
    seconds = (total_ms % 60_000) // 1000
    ms = total_ms % 1000
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}:{ms:03d}"


def parse_interval_text(text: str) -> Optional[int]:
    parts = [part.strip() for part in text.strip().split(":")]
    if len(parts) != 4:
        return None
    try:
        hh, mm, ss, ms = [int(part) for part in parts]
    except ValueError:
        return None
    if hh < 0 or mm < 0 or ss < 0 or ms < 0:
        return None
    if mm > 59 or ss > 59 or ms > 999:
        return None
    total_ms = ((hh * 60 + mm) * 60 + ss) * 1000 + ms
    return total_ms


class WaveformPreview(QWidget):
    def __init__(self, parent: Optional[QWidget] = None) -> None:
        super().__init__(parent)
        self.period_ms = 10.0
        self.pulse_ms = 4.0
        self.valid = True
        self.message = ""
        self.setMinimumHeight(220)
        self.border_color = QColor(0, 255, 140)
        self.grid_color = QColor(0, 110, 60)
        self.text_color = QColor(130, 255, 180)
        self.bg_color = QColor(8, 16, 10)

    def set_parameters(self, period_ms: float, pulse_ms: float, valid: bool, message: str) -> None:
        self.period_ms = max(0.001, period_ms)
        self.pulse_ms = max(0.0, pulse_ms)
        self.valid = valid
        self.message = message
        self.update()

    def paintEvent(self, event) -> None:  # noqa: N802
        painter = QPainter(self)
        painter.setRenderHint(QPainter.RenderHint.Antialiasing)

        outer = self.rect().adjusted(8, 8, -8, -8)
        painter.fillRect(outer, self.bg_color)

        title_y = outer.top() + 20
        frame = outer.adjusted(0, 34, 0, 0)

        border_pen = QPen(self.border_color)
        border_pen.setWidth(2)
        painter.setPen(border_pen)
        painter.drawRect(frame)

        text_pen = QPen(self.text_color)
        painter.setPen(text_pen)
        painter.drawText(
            outer.left() + 14,
            title_y,
            f"Preview: 3 periods | period = {self.period_ms:.3f} ms | pulse width = {self.pulse_ms:.3f} ms",
        )

        left = frame.left() + 22
        right = frame.right() - 22
        top = frame.top() + 22
        bottom = frame.bottom() - 28
        width = max(10, right - left)

        if not self.valid:
            painter.drawText(left, top + 24, self.message)
            return

        y_high = top + 16
        y_low = bottom - 12
        period_px = width / 3.0

        grid_pen = QPen(self.grid_color)
        grid_pen.setStyle(Qt.PenStyle.DotLine)
        grid_pen.setWidth(1)
        painter.setPen(grid_pen)
        painter.drawLine(left, y_high, right, y_high)
        painter.drawLine(left, y_low, right, y_low)

        wave_pen = QPen(self.border_color)
        wave_pen.setWidth(2)
        painter.setPen(wave_pen)

        if math.isclose(self.pulse_ms, self.period_ms, rel_tol=0.0, abs_tol=1e-9):
            painter.drawLine(left, y_high, right, y_high)
        else:
            high_fraction = self.pulse_ms / self.period_ms
            x = float(left)
            for _ in range(3):
                x_start = x
                x_high_end = x_start + period_px * high_fraction
                x_period_end = x_start + period_px
                painter.drawLine(int(x_start), y_low, int(x_start), y_high)
                painter.drawLine(int(x_start), y_high, int(x_high_end), y_high)
                painter.drawLine(int(x_high_end), y_high, int(x_high_end), y_low)
                painter.drawLine(int(x_high_end), y_low, int(x_period_end), y_low)
                x = x_period_end

        painter.setPen(text_pen)
        painter.drawText(left, y_high - 8, "HIGH")
        painter.drawText(left, bottom + 10, "LOW")
        if math.isclose(self.pulse_ms, self.period_ms, rel_tol=0.0, abs_tol=1e-9):
            painter.drawText(max(left + 120, right - 235), bottom + 10, "Pulse width = period (constant HIGH)")
        else:
            painter.drawText(max(left + 120, right - 235), bottom + 10, f"Low phase = {self.period_ms - self.pulse_ms:.3f} ms")


class MainWindow(QMainWindow):
    def __init__(self) -> None:
        super().__init__()
        self.setWindowTitle("DeepEcoHAB Hardware Trigger / TTL Scheduler")
        self.serial_port: Optional[serial.Serial] = None
        self.rx_buffer = ""
        self.last_exported_paths: list[str] = []
        self.last_sent_command = ""
        self.last_fallback_command = ""
        self.fallback_attempted = False

        central = QWidget()
        self.setCentralWidget(central)
        root = QHBoxLayout(central)
        root.setContentsMargins(10, 10, 10, 10)
        root.setSpacing(10)

        left_col = QVBoxLayout()
        right_col = QVBoxLayout()
        root.addLayout(left_col, 1)
        root.addLayout(right_col, 1)

        ttl_group = QGroupBox("TTL parameters")
        ttl_form = QFormLayout(ttl_group)

        self.port_edit = QLineEdit("COM12")

        self.baud_spin = QSpinBox()
        self.baud_spin.setRange(300, 3_000_000)
        self.baud_spin.setValue(115200)

        self.mode_combo = QComboBox()
        self.mode_combo.addItems(["Frequency (Hz)", "Interval (HH:MM:SS:MS)"])

        self.freq_spin = QDoubleSpinBox()
        self.freq_spin.setDecimals(6)
        self.freq_spin.setRange(MIN_FREQ_HZ, 100000.0)
        self.freq_spin.setSingleStep(0.1)
        self.freq_spin.setValue(100.0)

        self.interval_edit = QLineEdit("00:00:00:010")
        self.interval_edit.setPlaceholderText("HH:MM:SS:MS")
        self.interval_edit.setEnabled(False)

        self.pulse_spin = QDoubleSpinBox()
        self.pulse_spin.setDecimals(3)
        self.pulse_spin.setRange(0.001, MAX_INTERVAL_MS)
        self.pulse_spin.setSingleStep(1.0)
        self.pulse_spin.setValue(4.0)
        self.pulse_spin.setSuffix(" ms")

        self.count_spin = QSpinBox()
        self.count_spin.setRange(1, 2_000_000_000)
        self.count_spin.setValue(100000)

        ttl_form.addRow("Port", self.port_edit)
        ttl_form.addRow("Baud", self.baud_spin)
        ttl_form.addRow("Timing mode", self.mode_combo)
        ttl_form.addRow("Frequency (Hz)", self.freq_spin)
        ttl_form.addRow("Interval (HH:MM:SS:MS)", self.interval_edit)
        ttl_form.addRow("Pulse width (ms)", self.pulse_spin)
        ttl_form.addRow("Pulse count", self.count_spin)

        sched_group = QGroupBox("Schedule")
        sched_layout = QGridLayout(sched_group)

        self.now_time_edit = QTimeEdit()
        self.now_time_edit.setDisplayFormat("HH:mm:ss")
        self.now_time_edit.setTime(QTime.currentTime())

        self.start_time_edit = QTimeEdit()
        self.start_time_edit.setDisplayFormat("HH:mm:ss")
        self.start_time_edit.setTime(QTime.currentTime().addSecs(60))

        self.use_pc_time_btn = QPushButton("Use PC time now")
        self.use_pc_time_btn.clicked.connect(self.use_pc_time_now)

        self.refresh_pc_time_on_arm_check = QCheckBox("Refresh current time from PC when arming")
        self.refresh_pc_time_on_arm_check.setChecked(True)

        sched_layout.addWidget(QLabel("Current time sent to Arduino"), 0, 0)
        sched_layout.addWidget(self.now_time_edit, 0, 1)
        sched_layout.addWidget(self.use_pc_time_btn, 0, 2)
        sched_layout.addWidget(QLabel("Scheduled start time"), 1, 0)
        sched_layout.addWidget(self.start_time_edit, 1, 1)
        sched_layout.addWidget(self.refresh_pc_time_on_arm_check, 2, 0, 1, 3)

        derived_group = QGroupBox("Derived values")
        derived_layout = QVBoxLayout(derived_group)
        self.duration_label = QLabel()
        self.delay_label = QLabel()
        self.constraint_label = QLabel()
        self.command_label = QLabel()
        derived_layout.addWidget(self.duration_label)
        derived_layout.addWidget(self.delay_label)
        derived_layout.addWidget(self.constraint_label)
        derived_layout.addWidget(self.command_label)

        left_col.addWidget(ttl_group)
        left_col.addWidget(sched_group)
        left_col.addWidget(derived_group)
        left_col.addStretch(1)

        wave_group = QGroupBox("Waveform")
        wave_layout = QVBoxLayout(wave_group)
        self.waveform = WaveformPreview()
        wave_layout.addWidget(self.waveform)

        export_group = QGroupBox("Export settings")
        export_layout = QGridLayout(export_group)
        self.export_dir_edit = QLineEdit(str(Path.home() / "Downloads"))
        self.export_browse_btn = QPushButton("Browse")
        self.export_now_btn = QPushButton("Export JSON + CSV now")
        self.auto_export_check = QCheckBox("Auto-export on arm")
        self.auto_export_check.setChecked(True)

        export_layout.addWidget(QLabel("Directory"), 0, 0)
        export_layout.addWidget(self.export_dir_edit, 0, 1)
        export_layout.addWidget(self.export_browse_btn, 0, 2)
        export_layout.addWidget(self.auto_export_check, 1, 0, 1, 2)
        export_layout.addWidget(self.export_now_btn, 1, 2)

        runtime_group = QGroupBox("Runtime")
        runtime_layout = QVBoxLayout(runtime_group)
        self.status_label = QLabel("Idle")
        self.arduino_time_label = QLabel("Arduino time: --:--:--")
        self.log_box = QTextEdit()
        self.log_box.setReadOnly(True)
        button_row = QHBoxLayout()
        self.arm_btn = QPushButton("Arm protocol")
        self.disconnect_btn = QPushButton("Disconnect")
        button_row.addWidget(self.arm_btn)
        button_row.addWidget(self.disconnect_btn)
        runtime_layout.addWidget(self.status_label)
        runtime_layout.addWidget(self.arduino_time_label)
        runtime_layout.addLayout(button_row)
        runtime_layout.addWidget(self.log_box)

        right_col.addWidget(wave_group)
        right_col.addWidget(export_group)
        right_col.addWidget(runtime_group, 1)

        self.mode_combo.currentIndexChanged.connect(self.update_timing_mode_widgets)
        self.freq_spin.valueChanged.connect(self.update_derived_values)
        self.pulse_spin.valueChanged.connect(self.update_derived_values)
        self.count_spin.valueChanged.connect(self.update_derived_values)
        self.now_time_edit.timeChanged.connect(self.update_derived_values)
        self.start_time_edit.timeChanged.connect(self.update_derived_values)
        self.interval_edit.textChanged.connect(self.update_derived_values)
        self.export_browse_btn.clicked.connect(self.browse_export_dir)
        self.export_now_btn.clicked.connect(self.export_settings_now)
        self.arm_btn.clicked.connect(self.arm_protocol)
        self.disconnect_btn.clicked.connect(self.disconnect_serial)

        self.poll_timer = QTimer(self)
        self.poll_timer.timeout.connect(self.poll_serial)
        self.poll_timer.start(100)

        self.update_timing_mode_widgets()
        self.update_derived_values()
        self.append_log(f"GUI started using {QT_API}.")

    def append_log(self, text: str) -> None:
        stamp = datetime.now().strftime("[%H:%M:%S]")
        self.log_box.append(f"{stamp} {text}")

    def use_pc_time_now(self) -> None:
        self.now_time_edit.setTime(QTime.currentTime())

    def browse_export_dir(self) -> None:
        directory = QFileDialog.getExistingDirectory(self, "Choose export directory", self.export_dir_edit.text().strip())
        if directory:
            self.export_dir_edit.setText(directory)

    def current_delay_seconds(self) -> int:
        now_secs = self.now_time_edit.time().hour() * 3600 + self.now_time_edit.time().minute() * 60 + self.now_time_edit.time().second()
        start_secs = self.start_time_edit.time().hour() * 3600 + self.start_time_edit.time().minute() * 60 + self.start_time_edit.time().second()
        if start_secs < now_secs:
            start_secs += 86400
        return start_secs - now_secs

    def update_timing_mode_widgets(self) -> None:
        use_freq = self.mode_combo.currentIndex() == 0
        self.freq_spin.setEnabled(use_freq)
        self.interval_edit.setEnabled(not use_freq)
        self.update_derived_values()

    def build_timing_config(self) -> TimingConfig:
        mode_name = self.mode_combo.currentText()
        pulse_ms = float(self.pulse_spin.value())
        count = int(self.count_spin.value())
        now_str = self.now_time_edit.time().toString("HH:mm:ss")
        start_str = self.start_time_edit.time().toString("HH:mm:ss")

        if self.mode_combo.currentIndex() == 0:
            freq_hz = float(self.freq_spin.value())
            interval_ms = 1000.0 / freq_hz
        else:
            interval_ms_parsed = parse_interval_text(self.interval_edit.text())
            if interval_ms_parsed is None:
                raise ValueError("Interval must be in HH:MM:SS:MS format.")
            if interval_ms_parsed < 1:
                raise ValueError("Interval must be at least 1 ms.")
            if interval_ms_parsed > MAX_INTERVAL_MS:
                raise ValueError("Maximum supported interval is 00:05:00:000.")
            interval_ms = float(interval_ms_parsed)
            freq_hz = 1000.0 / interval_ms

        if pulse_ms > interval_ms + 1e-9:
            raise ValueError(f"Pulse width ({pulse_ms:.3f} ms) must not exceed the period ({interval_ms:.3f} ms).")

        period_us = int(round(interval_ms * 1000.0))
        pulse_us = int(round(pulse_ms * 1000.0))
        low_phase_ms = max(0.0, interval_ms - pulse_ms)
        total_duration_s = count * interval_ms / 1000.0

        classic_arm_compatible = (
            self.mode_combo.currentIndex() == 0
            and abs(freq_hz - round(freq_hz)) < 1e-9
            and abs(pulse_ms - round(pulse_ms)) < 1e-9
            and freq_hz >= 1.0
            and pulse_ms <= interval_ms + 1e-9
        )

        return TimingConfig(
            mode_name=mode_name,
            period_us=period_us,
            pulse_us=pulse_us,
            count=count,
            freq_hz=freq_hz,
            interval_ms=interval_ms,
            total_duration_s=total_duration_s,
            low_phase_ms=low_phase_ms,
            now_str=now_str,
            start_str=start_str,
            classic_arm_compatible=classic_arm_compatible,
        )

    def update_derived_values(self) -> None:
        try:
            cfg = self.build_timing_config()
            self.duration_label.setText(f"Total stimulation time = {seconds_to_hms(cfg.total_duration_s)}")
            self.delay_label.setText(f"Delay until start = {seconds_to_hms(self.current_delay_seconds())}")
            if math.isclose(cfg.pulse_us, cfg.period_us, rel_tol=0.0, abs_tol=1):
                msg = f"Valid waveform: period = {cfg.interval_ms:.3f} ms, pulse width = period, output stays HIGH during stimulation."
            else:
                msg = f"Valid waveform: period = {cfg.interval_ms:.3f} ms, low phase = {cfg.low_phase_ms:.3f} ms."
            self.constraint_label.setText(msg)
            cmd_preview = f"ARMUS {cfg.period_us} {cfg.pulse_us} {cfg.count} {cfg.now_str} {cfg.start_str}"
            if cfg.classic_arm_compatible:
                classic = f"ARM {int(round(cfg.freq_hz))} {int(round(cfg.pulse_us / 1000.0))} {cfg.count} {cfg.now_str} {cfg.start_str}"
                self.command_label.setText(f"Primary command: {cmd_preview} | Fallback: {classic}")
            else:
                self.command_label.setText(f"Primary command: {cmd_preview}")
            self.waveform.set_parameters(cfg.interval_ms, cfg.pulse_us / 1000.0, True, "")
        except ValueError as exc:
            self.duration_label.setText("Total stimulation time = --:--:--")
            self.delay_label.setText(f"Delay until start = {seconds_to_hms(self.current_delay_seconds())}")
            self.constraint_label.setText(f"Invalid: {exc}")
            self.command_label.setText("Primary command: --")
            pulse_ms = float(self.pulse_spin.value())
            interval_ms = 10.0
            if self.mode_combo.currentIndex() == 0 and self.freq_spin.value() > 0:
                interval_ms = 1000.0 / float(self.freq_spin.value())
            self.waveform.set_parameters(interval_ms, pulse_ms, False, str(exc))

    def export_settings_now(self) -> None:
        try:
            cfg = self.build_timing_config()
        except ValueError as exc:
            QMessageBox.warning(self, "Invalid settings", str(exc))
            return

        try:
            paths = self.export_settings(cfg)
        except Exception as exc:  # noqa: BLE001
            QMessageBox.critical(self, "Export failed", str(exc))
            return

        self.append_log("Exported settings to:")
        for path in paths:
            self.append_log(str(path))

    def export_settings(self, cfg: TimingConfig) -> list[Path]:
        export_dir = Path(self.export_dir_edit.text().strip()).expanduser()
        export_dir.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        base = export_dir / f"ttl_settings_{stamp}"
        json_path = base.with_suffix(".json")
        csv_path = base.with_suffix(".csv")

        payload = asdict(cfg)
        payload.update(
            {
                "exported_at": datetime.now().isoformat(timespec="seconds"),
                "delay_until_start_s": self.current_delay_seconds(),
                "delay_until_start_hms": seconds_to_hms(self.current_delay_seconds()),
                "interval_text": format_interval_ms(int(round(cfg.interval_ms))),
                "refresh_pc_time_on_arm": self.refresh_pc_time_on_arm_check.isChecked(),
            }
        )

        with json_path.open("w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)

        with csv_path.open("w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["field", "value"])
            for key, value in payload.items():
                writer.writerow([key, value])

        self.last_exported_paths = [str(json_path), str(csv_path)]
        return [json_path, csv_path]

    def open_serial(self) -> serial.Serial:
        port = self.port_edit.text().strip()
        baud = self.baud_spin.value()
        ser = serial.Serial(port, baud, timeout=0, write_timeout=1)
        time.sleep(2.0)
        ser.reset_input_buffer()
        ser.reset_output_buffer()
        return ser

    def arm_protocol(self) -> None:
        if self.refresh_pc_time_on_arm_check.isChecked():
            self.now_time_edit.setTime(QTime.currentTime())
            self.append_log("Current time refreshed from PC immediately before arming.")

        try:
            cfg = self.build_timing_config()
        except ValueError as exc:
            QMessageBox.warning(self, "Invalid parameters", str(exc))
            return

        if self.auto_export_check.isChecked():
            try:
                paths = self.export_settings(cfg)
                self.append_log("Exported settings to:")
                for path in paths:
                    self.append_log(str(path))
            except Exception as exc:  # noqa: BLE001
                QMessageBox.critical(self, "Export failed", str(exc))
                return

        self.disconnect_serial(silent=True)
        try:
            self.serial_port = self.open_serial()
        except SerialException as exc:
            QMessageBox.critical(self, "Serial error", str(exc))
            self.status_label.setText("Failed to open serial port")
            return

        self.fallback_attempted = False
        self.last_sent_command = f"ARMUS {cfg.period_us} {cfg.pulse_us} {cfg.count} {cfg.now_str} {cfg.start_str}"
        self.last_fallback_command = ""
        if cfg.classic_arm_compatible:
            self.last_fallback_command = (
                f"ARM {int(round(cfg.freq_hz))} {int(round(cfg.pulse_us / 1000.0))} {cfg.count} {cfg.now_str} {cfg.start_str}"
            )

        try:
            self.serial_port.write((self.last_sent_command + "\n").encode("ascii"))
            self.serial_port.flush()
        except SerialException as exc:
            QMessageBox.critical(self, "Serial write error", str(exc))
            self.status_label.setText("Failed to send ARM command")
            self.disconnect_serial(silent=True)
            return

        self.status_label.setText(f"Command sent. Waiting for scheduled start at {cfg.start_str}.")
        self.arduino_time_label.setText("Arduino time: waiting for first update...")
        self.append_log(f"> {self.last_sent_command}")
        if self.last_fallback_command:
            self.append_log("Classic ARM fallback is available if firmware rejects ARMUS.")

    def disconnect_serial(self, silent: bool = False) -> None:
        if self.serial_port is not None:
            try:
                self.serial_port.close()
            except Exception:
                pass
            finally:
                self.serial_port = None
        if not silent:
            self.status_label.setText("Disconnected")
            self.append_log("Serial port closed.")

    def poll_serial(self) -> None:
        if self.serial_port is None:
            return
        try:
            incoming = self.serial_port.read_all()
        except SerialException as exc:
            self.append_log(f"Serial read error: {exc}")
            self.status_label.setText("Serial read error")
            self.disconnect_serial(silent=True)
            return
        if not incoming:
            return
        self.rx_buffer += incoming.decode("ascii", errors="ignore")
        while "\n" in self.rx_buffer:
            line, self.rx_buffer = self.rx_buffer.split("\n", 1)
            self.handle_serial_line(line.strip())

    def try_classic_fallback(self) -> None:
        if self.serial_port is None or not self.last_fallback_command or self.fallback_attempted:
            return
        self.fallback_attempted = True
        try:
            self.serial_port.write((self.last_fallback_command + "\n").encode("ascii"))
            self.serial_port.flush()
            self.append_log(f"> {self.last_fallback_command}")
            self.append_log("ARMUS was rejected. Sent classic ARM fallback.")
        except SerialException as exc:
            self.append_log(f"Fallback serial write error: {exc}")

    def handle_serial_line(self, line: str) -> None:
        if not line:
            return
        self.append_log(line)
        if line == "READY":
            self.status_label.setText("Arduino ready")
        elif line.startswith("ARMED "):
            self.status_label.setText(line)
        elif line.startswith("TIME "):
            clock_text = line[5:].strip()
            self.arduino_time_label.setText(f"Arduino time: {clock_text}")
            self.status_label.setText("Arduino clock running")
        elif line == "START":
            self.status_label.setText("Stimulation started")
        elif line == "DONE":
            self.status_label.setText("Stimulation finished")
        elif line == "ERR CMD":
            self.status_label.setText("Arduino error: ERR CMD")
            self.try_classic_fallback()
        elif line.startswith("ERR"):
            self.status_label.setText(f"Arduino error: {line}")

    def closeEvent(self, event) -> None:  # noqa: N802
        self.disconnect_serial(silent=True)
        super().closeEvent(event)


def apply_dark_theme(app: QApplication) -> None:
    """DeepEcoHAB dark theme, matched to the main menu and RFID DAQ GUI."""
    app.setStyle("Fusion")
    palette = QPalette()
    palette.setColor(QPalette.ColorRole.Window, QColor("#0f1115"))
    palette.setColor(QPalette.ColorRole.WindowText, QColor("#e6e6e6"))
    palette.setColor(QPalette.ColorRole.Base, QColor("#12151c"))
    palette.setColor(QPalette.ColorRole.AlternateBase, QColor("#0f1115"))
    palette.setColor(QPalette.ColorRole.ToolTipBase, QColor("#0f1115"))
    palette.setColor(QPalette.ColorRole.ToolTipText, QColor("#e6e6e6"))
    palette.setColor(QPalette.ColorRole.Text, QColor("#e6e6e6"))
    palette.setColor(QPalette.ColorRole.Button, QColor("#151a23"))
    palette.setColor(QPalette.ColorRole.ButtonText, QColor("#e6e6e6"))
    palette.setColor(QPalette.ColorRole.BrightText, QColor("#ffffff"))
    palette.setColor(QPalette.ColorRole.Highlight, QColor("#2d6cdf"))
    palette.setColor(QPalette.ColorRole.HighlightedText, QColor("#ffffff"))
    app.setPalette(palette)

    app.setStyleSheet(
        """
        QWidget {
            background: #0f1115;
            color: #e6e6e6;
            font-family: Segoe UI, Arial;
            font-size: 10pt;
        }
        QLabel, QCheckBox, QRadioButton {
            background: transparent;
            color: #e6e6e6;
        }
        QToolTip {
            color: #e6e6e6;
            background-color: #0f1115;
            border: 1px solid #2b2f3a;
        }
        QGroupBox {
            background: #151a23;
            border: 1px solid #2b2f3a;
            border-radius: 8px;
            margin-top: 12px;
            padding: 10px;
            font-weight: 600;
            color: #e6e6e6;
        }
        QGroupBox::title {
            subcontrol-origin: margin;
            left: 10px;
            padding: 0 4px;
            color: #e6e6e6;
            background: transparent;
        }
        QLineEdit, QSpinBox, QDoubleSpinBox, QTimeEdit, QTextEdit, QComboBox {
            background: #12151c;
            color: #e6e6e6;
            border: 1px solid #2b2f3a;
            border-radius: 6px;
            padding: 5px 7px;
            min-height: 26px;
            selection-background-color: #2d6cdf;
        }
        QTextEdit {
            font-family: Consolas, Courier New, monospace;
        }
        QPushButton {
            background: #1a2230;
            color: #e6e6e6;
            border: 1px solid #2b2f3a;
            border-radius: 8px;
            padding: 7px 12px;
            min-height: 30px;
            font-weight: 600;
        }
        QPushButton:hover {
            background: #22304a;
            border: 1px solid #2d6cdf;
        }
        QPushButton:pressed {
            background: #2d6cdf;
            color: #ffffff;
        }
        QPushButton:disabled {
            background: #11151d;
            color: #7c7c7c;
        }
        QCheckBox {
            color: #e6e6e6;
            spacing: 8px;
        }
        QLabel {
            color: #e6e6e6;
        }
        """
    )


def main() -> None:
    app = QApplication(sys.argv)
    apply_dark_theme(app)
    window = MainWindow()
    window.resize(1280, 840)
    window.show()
    sys.exit(app.exec())


if __name__ == "__main__":
    main()
